# [INSTRUCTIONS]

Note we utilize duckdb view chains to modify the large dataset

If you are unfamiliar with duckdb please refer to the documentation: https://duckdb.org/docs/stable/clients/python/overview

For the first run, ensure the globals snellius = True create_dataset = True, create_graph_features = True, after data creation both can be turned off
For all subsequent always set snellius = True

DOWNLOAD: This file must be moved from Snellius to its corresponding local folder. Ignore this tag if you're able to run the notebook locally.

Set the following globals to true for first run and download the files and folder



DOWNLOAD: downloads/data/filtered/final_aggregated/services_monthly.parquet

DOWNLOAD: downloads/data/filtered/final_aggregated/services_monthly_graph_features.parquet

DOWNLOAD: (folder) downloads/data/filtered/graph_features_output/

DOWNLOAD:  downloads/data/filtered/final_aggregated/final_dataset_with_operational.parquet


# Data Loading

## GLOBALS

In [1]:
snellius = True  # Set to True if running on Snellius, False for local development
partition = "operational_features_process" if snellius else "subset_week"

In [2]:
create_dataset = True  # Set to True to create the dataset, False to skip for faster execution
create_graph_features = True # Set to True to create the graph features dataset 

#MISC
check_stats = False  # cSet to True to display intermediate stats, False to skip for faster execution
create_intermediate_datasets = False
check_rowc = False


In [3]:
rstats = (snellius is True) and (check_stats is True) #run stats
cdata_new = (snellius is True) and (create_dataset is True) # create data
cdata_old = (cdata_new is True) and (create_intermediate_datasets is True) 

## Set root folder path 

In [4]:
# pip install any missing packages before running the notebook
import duckdb
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx

In [ ]:
import os
from pathlib import Path

if snellius is False:
    # Change later to snellius $HOME or whatever folder you want to work in
    target_root = Path(r"C:\Users\jialo\Desktop\MSc-DS-Thesis\MSc-Thesis-Repo\MSc-Thesis\Bao")

    # Check if it exists before moving
    if target_root.exists():
        os.chdir(target_root)
        print(f"✅ Success! Moved to: {Path.cwd()}")
    else:
        print(f"❌ Error: The folder '{target_root}' does not exist.")


## Snellius Setup
We can directly read and write files to our personal $HOME node on snellius (instead of the tmpdir and copying back and forth)

To run this notebook, utilize the same project structure from /Bao onwards

First transfer files to Snellius (vscode) terminal:
scp -r ./MSc-Thesis/Bao/X username@snellius.surf.nl:~/Project_folder/X
where X = {all_code, downloads, media} folders

(or gitbash):
rsync -avP ./MSc-Thesis/Bao/{all_code,downloads,media} name@snellius.surf.nl:~/NS_Thesis/

On Snellius:
setup environment.yml
run sbatch InstallEnv.job
run the corresponding .job

In [ ]:
if snellius is True:
    from pathlib import Path
    # Get the home directory path object
    home_dir = Path.home() 

    # Build a path to your code
    target_root = home_dir / "NS_Thesis"

    print(target_root)

    # Test writing a file to confirm the path works
    # # 1. Define the path (Home Directory)
    # # On Snellius, this will resolve to /home/jbao
    # file_path = Path.home() / "test_file.txt"

    # # 2. Write a small test message
    # try:
    #     with open(file_path, "w") as f:
    #         f.write("This is a test file to verify the home directory path.\n")
        
    #     print(f"✅ Success! File saved to: {file_path}")
    #     print("⚠️  Reminder: Do not save large datasets here (16GB Quota).")
    # except Exception as e:
    #     print(f"❌ Error: Could not write to file. {e}")

/home/jbao/NS_Thesis


## Set paths

In [ ]:
project_root = target_root

root_folder =  project_root / "downloads/data/raw/"
filtered_output_root_folder = project_root / "downloads/data/filtered/"

services_path = root_folder / "NS-services/services_merged_all_ns_only.parquet"
disruptions_path = root_folder / "NS-disruptions/disruptions_merged_all.parquet"
stations_path = root_folder / "NS-stations/stations-2023-09-nl.csv" #duckdb seem to handle 'NA' station code properly, so load the original file
station_distances_path = root_folder / "NS-tariff-distances/tariff-distances-2022-01.csv" #probably not used in eda, might only be useful for graph edges feature
stations_connections_path = root_folder / "railway_map/connection_edges.parquet" 

weather_path = root_folder / "weather/weather_merged_all.parquet"
holiday_path = root_folder / "holidays/dutch_holidays_2019_2025.parquet"

media_folder = project_root / "media"
# if not media_folder.exists():
#     media_folder.mkdir()

# 1. Setup paths to iterate over

paths = {
    "Services": services_path,
    "Disruptions": disruptions_path,
    "Stations": stations_path,
    "Station Distances": station_distances_path,
    "Weather": weather_path,
    "Holidays": holiday_path
}


# 2. Paths of filtered datasets (after EDA and cleaning)
aggregated_monthly_path = filtered_output_root_folder / "final_aggregated/services_monthly.parquet"

GRAPH_OUTPUT_DIR = filtered_output_root_folder / "graph_features_output"
os.makedirs(GRAPH_OUTPUT_DIR, exist_ok=True)

graph_features_path = filtered_output_root_folder / "final_aggregated/services_monthly_graph_features.parquet"

In [ ]:
## Connect to database
db_name = 'main_thesis_data.duckdb'

# Check before
print(f"Does file exist? {os.path.exists(db_name)}")

con = duckdb.connect(database='main_thesis_data.duckdb') 

# Explicitly cap RAM so DuckDB knows WHEN to start spilling
# (e.g., set to 70-80% of your actual machine RAM)
if snellius is False:
    con.execute("SET memory_limit='5GB'")   

Does file exist? True


## SAVING CODE

duckdb tables to parquet

In [9]:
# aggregated_hourly = filtered_output_root_folder / "final_aggregated/services_hourly.parquet"
# con.execute(f"COPY services_hourly_agg TO '{aggregated_hourly}' (FORMAT PARQUET)")
# print("Done! Saved as ", aggregated_hourly)

## Helper functions

In [ ]:
def check_arrival_delay_distribution(dataset_name):
    print(f"Checking arrival_delay_min distribution for {dataset_name}...")
    ad_distribution_df = con.execute(f"""
        SELECT 
            -- Create the bin floor (0, 100, 200...)
            FLOOR(arrival_delay_min / 100) * 100 AS bin_start,
            
            -- Create a readable label
            CAST(FLOOR(arrival_delay_min / 100) * 100 AS INTEGER) || ' to ' || 
            CAST(FLOOR(arrival_delay_min / 100) * 100 + 100 AS INTEGER) || ' min' AS bin_label,
            
            -- Count services in this bin
            COUNT(*) AS count,
            
            -- Percentage
            ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM {dataset_name} WHERE arrival_delay_min IS NOT NULL), 4) AS pct
            
        FROM {dataset_name}
        WHERE arrival_delay_min IS NOT NULL
        GROUP BY 1, 2
        ORDER BY 1
    """).df()

    return display(ad_distribution_df)

def count_row_diff(data_before, data_after):
    print(f"Checking rows for {data_before} and {data_after}...")

    rows_df = con.execute(f"""
        SELECT 
            (SELECT COUNT(*) FROM {data_before}) AS original_rows,
            (SELECT COUNT(*) FROM {data_after}) AS remaining_rows,
            (SELECT COUNT(*) FROM {data_before}) - (SELECT COUNT(*) FROM {data_after}) AS dropped_rows
    """).df()
    return display(rows_df)
    

## Duckdb prep

In [ ]:
# 2. Initialize Disk-Based Database
# This creates a file 'main_thesis_data.duckdb' in your current folder.
# Intermediate calculations (joins/group bys) will spill here instead of crashing RAM.
# con = duckdb.connect(database='main_thesis_data.duckdb') 

# 3. Register Views (Zero-Copy), Run once
# We use VIEWs so we don't duplicate the Parquet data into the .duckdb file.
# DuckDB reads directly from the parquet files on demand.
con.execute(f"CREATE OR REPLACE VIEW raw_services AS SELECT * FROM read_parquet('{services_path}')")
con.execute(f"CREATE OR REPLACE VIEW raw_disruptions AS SELECT * FROM read_parquet('{disruptions_path}')")
con.execute(f"CREATE OR REPLACE VIEW raw_weather AS SELECT * FROM read_parquet('{weather_path}')")
con.execute(f"CREATE OR REPLACE VIEW stations AS SELECT * FROM read_csv_auto('{stations_path}')")
con.execute(f"CREATE OR REPLACE VIEW raw_holidays AS SELECT * FROM read_parquet('{holiday_path}')")
con.execute(f"CREATE OR REPLACE VIEW raw_station_distances AS SELECT * FROM read_csv_auto('{station_distances_path}')")
con.execute(f"CREATE OR REPLACE VIEW raw_stations_connections AS SELECT * FROM read_parquet('{stations_connections_path}')") # Prob not needed as stations from RdT is pulled from NS API already, but just in case

# 1. Convert your paths to Path objects (works safely even if they are currently strings)
agg_path = Path(aggregated_monthly_path)
graph_path = Path(graph_features_path)

if agg_path.exists():
    con.execute(f"CREATE OR REPLACE VIEW services_monthly AS SELECT * FROM read_parquet('{aggregated_monthly_path}')")
    print(f"Success: Created view 'services_monthly'")
else:
    print(f"Warning: File not found -> {agg_path}. Skipping view creation. Set GLOBAL variable create_dataset = True")
    
if graph_path.exists():
    con.execute(f"CREATE OR REPLACE VIEW services_monthly_graph_features AS SELECT * FROM read_parquet('{graph_features_path}')")
    print(f"Success: Created view 'services_monthly_graph_features'")
else:
    print(f"Warning: File not found -> {graph_path}. Skipping view creation. Set GLOBAL variable create_graph_features = True")



# 4. Create CLEAN Views (Logic remains the same, but executed safely)
con.execute("""
    CREATE OR REPLACE VIEW services AS 
    SELECT 
        "Service:RDT-ID" AS service_id,
        "Service:Date" AS service_date,
        "Service:Type" AS train_type,
            
        "Service:Completely cancelled" AS is_completely_cancelled, 
        "Service:Partly cancelled" AS is_partly_cancelled,

        "Stop:Station code" AS station_code,
        "Stop:Station name" AS station_name,
        
        -- Apply Timezone conversion HERE in the SELECT statement
        ("Stop:Arrival time" AT TIME ZONE 'Europe/Amsterdam')::TIMESTAMP AS arrival_time,
        
        "Stop:Arrival delay" AS arrival_delay_min,
        "Stop:Arrival cancelled" AS is_arrival_cancelled,
        
        -- Apply Timezone conversion HERE too
        ("Stop:Departure time" AT TIME ZONE 'Europe/Amsterdam')::TIMESTAMP AS departure_time,
        
        "Stop:Departure delay" AS departure_delay_min,
        "Stop:Departure cancelled" AS is_departure_cancelled,
        
        "Stop:Platform change" AS has_platform_change,
            
    FROM raw_services
""")

print("✅ Table 'services' created with clean names and local timestamps.")

con.execute("""
    CREATE OR REPLACE VIEW disruptions AS 
    SELECT 
        *,
        CASE 
            WHEN rdt_station_codes LIKE '[%]' 
            THEN string_split(trim(rdt_station_codes, '[]'), ', ') 
            ELSE string_split(rdt_station_codes, ',') 
        END AS station_array
    FROM raw_disruptions
""")

print("Disk-based database initialized. Queries will now spill to 'main_thesis_data.duckdb' if needed.")

Success: Created view 'services_monthly'
Success: Created view 'services_monthly_graph_features'
✅ Table 'services' created with clean names and local timestamps.
Disk-based database initialized. Queries will now spill to 'main_thesis_data.duckdb' if needed.


In [12]:
con.execute("""DESCRIBE services_monthly_graph_features""").df()

,column_name,column_type,null,key,default,extra
0,source,VARCHAR,YES,None,None,None
1,target,VARCHAR,YES,None,None,None
2,YearMonth,VARCHAR,YES,None,None,None
3,year,BIGINT,YES,None,None,None
4,month,BIGINT,YES,None,None,None
5,distance,INTEGER,YES,None,None,None
6,total_departure_delay_minutes,DOUBLE,YES,None,None,None
7,avg_departure_delay_minutes,DOUBLE,YES,None,None,None
8,count_delayed_departure_services,BIGINT,YES,None,None,None
9,total_delay_minutes,DOUBLE,YES,None,None,None


## Total services in dataset

Total services ROWCOUNT = 83_816_585

In [ ]:
if rstats:
    display(con.execute("SELECT count(service_id) FROM services").df())

# Processing
from duckdb EDA

## Preprocess: NA values Cleaning and Handling
### NA in Services Arrival and Departures Columns
In short the meaning of NA values in services arrival and departure columns indicate either unscheduled services, a start station or terminating stations. Which is deduced from the rdt documentation "Each row in these files represent a stop at a station. Each service at least departs from and arrives at a station (i.e. two rows). For each stop, you can find the name of the station, the arrival and departure time, delays and cancellations. The exact meaning of each column is explained below." https://www.rijdendetreinen.nl/en/open-data/train-archive

In [ ]:
if rstats:
    # Check if there are cases where arrival_delay_min is present but arrival_time is NULL (which would mean that a train was delayed but not scheduled to arrive)
    display(con.execute("""
        SELECT 
            service_id,
            station_name,
            arrival_time,
            arrival_delay_min,
            is_arrival_cancelled
        FROM services
        WHERE 
            arrival_delay_min IS NOT NULL 
            AND arrival_time IS NULL
        LIMIT 20
    """).df())

In [ ]:
if rstats:
    # Query to find "Terminus" stops (Arrival exists, Departure missing)
    df_terminus = con.execute("""
        SELECT 
            service_id,
            station_name,
            arrival_delay_min,
            arrival_time,
            is_arrival_cancelled
        FROM services
        WHERE 
            -- Condition 1: Departure info is missing (OR logic as requested)
            (arrival_time IS NULL OR arrival_delay_min IS NULL OR is_arrival_cancelled IS NULL)
            
            AND 
            
            -- Condition 2: Arrival info is present (OR logic as requested)
            (departure_time IS NOT NULL OR departure_delay_min IS NOT NULL OR is_departure_cancelled IS NOT NULL)
        
        -- Limit to see a sample first
        LIMIT 20
    """).df()

    display(df_terminus)

#### p.1) Handling NA values in the services dataset
- Remove unscheduled trains (NA at both arrival and departure columns)
- Fill start stations (NA at arrival columns, it never 'arrives' so columns:arrival_time = departure_time, arrival_delay_min = 0, is_arrival_cancelled = FALSE)
- Removing end stations

We note that:

ALL NA values in both arrival and departure columns indicate a unscheduled service

ALL NA values in arrival_time and arrival_delay_min indicate the start of a service.

ALL NA values in departure_time and arrival_delay_min indicate the end of a service. 

A starting station could never have arrival delay as it is already there (IS NA values in arrival columns)

A end station could never have departure delay as will never depart there (IS NA values in departure columns)

By handling the NA values like this we add some biases to our dataset, but it logically follows from the schedule.

##### A) Removing unscheduled services (services with ONLY NA values in both departure and arrival time & delay) 
REMOVED: 5652 services

ROWCOUNT services_scheduled = 83_810_933

In [ ]:
con.execute("""
    CREATE OR REPLACE VIEW services_scheduled AS 
    SELECT * FROM services
    WHERE NOT (
        arrival_time IS NULL 
        AND departure_time IS NULL 
        AND arrival_delay_min IS NULL
        AND departure_delay_min IS NULL
    )
""")

print("✅ View 'services_scheduled' created (Unscheduled rows hidden).")

✅ View 'services_scheduled' created (Unscheduled rows hidden).


In [ ]:
if check_rowc:
    count_row_diff("services", "services_scheduled")

In [ ]:
if rstats:
    display(con.execute("SELECT count(service_id) FROM services_scheduled").df())

##### B) We impute the values for the starting station (NA values for ALL arrival columns) (8_121_562 services) and REMOVE the end stations (NA values for ALL departure columns) (8_119_198 services)

2364 train services seem to be missing end stations, this is most be explained by the column "Service:Train number" due to merging (or splitting) of trains, as the following is mentioned in the dataset documentation:
"A single service may sometimes have multiple train numbers. For example, when a train is split in two parts, or when a train changes a train number on a major station halfway." https://www.rijdendetreinen.nl/en/open-data/train-archive
We leave these in as we are analysing stop trajectories in our train railway network, not specific service train trajectories

In [ ]:
con.execute("""
    CREATE OR REPLACE VIEW services_filled AS 
    SELECT 
        * REPLACE (
            -- =========================================================
            -- 1. STRICT Fix for Start Stations (Fill Arrival)
            -- Only runs if ALL 3 arrival columns are NULL
            -- =========================================================
            CASE 
                WHEN arrival_time IS NULL 
                    AND arrival_delay_min IS NULL 
                    AND is_arrival_cancelled IS NULL
                THEN departure_time             -- Action: Fill with Departure
                ELSE arrival_time               -- Action: Keep original (even if NULL)
            END AS arrival_time,

            CASE 
                WHEN arrival_time IS NULL 
                    AND arrival_delay_min IS NULL 
                    AND is_arrival_cancelled IS NULL
                THEN 0                          -- Action: Assume 0 delay
                ELSE arrival_delay_min          -- Action: Keep original
            END AS arrival_delay_min,

            CASE 
                WHEN arrival_time IS NULL 
                    AND arrival_delay_min IS NULL 
                    AND is_arrival_cancelled IS NULL
                THEN FALSE                      -- Action: Assume not cancelled
                ELSE is_arrival_cancelled       -- Action: Keep original
            END AS is_arrival_cancelled
        )
    FROM services_scheduled 
    -- =========================================================
    -- 2. REMOVE rows where ALL departure columns are NULL
    -- =========================================================
    WHERE NOT (
        departure_time IS NULL 
        AND departure_delay_min IS NULL 
        AND is_departure_cancelled IS NULL
    )
""")

print("✅ View 'services_filled' created. Strict imputation for arrivals applied, and missing departures dropped.")

✅ View 'services_filled' created. Strict imputation for arrivals applied, and missing departures dropped.


Should be 8_119_198 rows dropped

In [ ]:
if check_rowc:
    count_row_diff("services_scheduled", "services_filled")

#### p.2) Arrival Delay of Cancellations

In short we remap the 7_813_345 cancelled services as delayed services

We set an arrival delay penalty of 30 min to partly cancelled services and 60 min to completely cancelled services

Cancellations as % of all services

In [ ]:
if rstats:
    con.execute("""
        SELECT 
            -- Total
            count(*) as total_services,
            
            -- Counts
            count(*) FILTER (WHERE is_partly_cancelled OR is_completely_cancelled) as total_all_cancelled_services,
            count(*) FILTER (WHERE is_partly_cancelled AND NOT is_completely_cancelled) as count_partly_cancelled,
            count(*) FILTER (WHERE is_completely_cancelled) as count_completely_cancelled,
            count(*) FILTER (WHERE is_partly_cancelled AND is_completely_cancelled) as count_both_cancelled,
                
                
            -- Percentages (Logic repeated)
            round(
                total_all_cancelled_services / total_services * 100.0, 
            2) as pct_all_cancelled,

            round(
                count_partly_cancelled / total_services * 100.0, 
            2) as pct_partly_cancelled,

            round(
                count_completely_cancelled / total_services * 100.0, 
            2) as pct_completely_cancelled,
                
            round(
                count_both_cancelled / total_services * 100.0,
            2) as pct_both_cancelled
            
        FROM services_filled
    """).df()

Cancellations as % of all CANCELLED services

In [ ]:
if rstats:
    con.execute("""
        SELECT 
            -- Total
            -- count(*) as total_services,
            
            -- Counts
            count(*) FILTER (WHERE is_partly_cancelled OR is_completely_cancelled) as total_all_cancelled_services,
            count(*) FILTER (WHERE is_partly_cancelled AND NOT is_completely_cancelled) as count_partly_cancelled,
            count(*) FILTER (WHERE is_completely_cancelled) as count_completely_cancelled,
            count(*) FILTER (WHERE is_partly_cancelled AND is_completely_cancelled) as count_both_cancelled,
                
            -- Percentages (Logic repeated)
            round(
                total_all_cancelled_services / total_all_cancelled_services * 100.0, 
            2) as pct_all_cancelled,

            round(
                count_partly_cancelled / total_all_cancelled_services * 100.0, 
            2) as pct_partly_cancelled,

            round(
                count_completely_cancelled / total_all_cancelled_services * 100.0, 
            2) as pct_completely_cancelled,
            
            round(
                count_both_cancelled / total_all_cancelled_services * 100.0,
            2) as pct_both_cancelled
        
            
        FROM services_filled
    """).df()

9.3% (7_813_345 million services) of all services (83_810_933) are affected by cancellations

out of all cancelled services

76.1% is partly cancelled

23.9% is completely cancelled

In [23]:
print(7_813_345 / 83_810_933 * 100)

9.322584441340128


In [ ]:
if rstats:
    con.execute("""
        SELECT 
            -- Total Cancelled Rows
            count(*) as total_cancelled,
            
            -- 1. Delay >= 60 mins
            count(*) FILTER (WHERE arrival_delay_min >= 60) as count_gt_60,
            round(count(*) FILTER (WHERE arrival_delay_min >= 60) * 100.0 / count(*), 2) as pct_gt_60,

            -- 2. Delay >= 30 < 60 mins
            count(*) FILTER (WHERE arrival_delay_min >= 30 AND arrival_delay_min < 60) as count_gt_30,
            round(count(*) FILTER (WHERE arrival_delay_min >= 30 AND arrival_delay_min < 60) * 100.0 / count(*), 2) as pct_gt_30,
            
            -- 2. Delay > 0 < 30 mins
            count(*) FILTER (WHERE arrival_delay_min > 0 AND arrival_delay_min < 30) as count_gt_1,
            round(count(*) FILTER (WHERE arrival_delay_min > 0 AND arrival_delay_min < 30) * 100.0 / count(*), 2) as pct_gt_1,

            -- 3. Delay <= 0
            count(*) FILTER (WHERE arrival_delay_min <= 0) as count_eq_0,
            round(count(*) FILTER (WHERE arrival_delay_min <= 0) * 100.0 / count(*), 2) as pct_eq_0,
                
        FROM services_filled
        WHERE 
            -- Focus only on cancelled services
            (is_partly_cancelled = TRUE OR is_completely_cancelled = TRUE)
    """).df()

##### p.2A) Set Arrival Delay Penalty for both completely and partly cancelled services (changes the delayed classification distribution)
The exact number doesnt matter, is significantly delay target label based on number of services on trajectory.
We essentially count these cancelled and partly cancelled services as delayed services, taking into account the passenger's perspective instead of completely dropping these relevant services

UNCOMMENT

In [25]:
# con.execute("""
# 	CREATE OR REPLACE VIEW services_penalty AS 
# 	SELECT * REPLACE (
# 		CASE
# 			WHEN is_completely_cancelled = TRUE THEN 60
# 			WHEN is_partly_cancelled = TRUE THEN 30
# 			ELSE arrival_delay_min
# 		END AS arrival_delay_min
# 	)
# 	FROM services_filled
# """)

# # Verify the "overwrite" worked
# if rstats:
# 	con.execute("SELECT is_completely_cancelled, arrival_delay_min FROM services_penalty WHERE is_completely_cancelled = TRUE LIMIT 5").df()

In [ ]:
con.execute("""
    CREATE OR REPLACE VIEW services_penalty AS 
    SELECT 
        * REPLACE (
            CASE
                WHEN is_completely_cancelled = TRUE THEN 60
                WHEN is_partly_cancelled = TRUE THEN 30
                ELSE arrival_delay_min
            END AS arrival_delay_min
        ),
        
        -- 1. True Total Services (Sum this later to get total scheduled)
        1 AS total_service_count,
        
        -- 2. Pure Arrival Delays (Delay > 0, but NOT cancelled)
        CASE 
            WHEN arrival_delay_min > 0 
             AND coalesce(is_completely_cancelled, FALSE) = FALSE 
             AND coalesce(is_partly_cancelled, FALSE) = FALSE 
            THEN 1 
            ELSE 0 
        END AS pure_delay_count,
        
        -- 3. Cancellations (Partly or Completely)
        CASE 
            WHEN is_completely_cancelled = TRUE 
              OR is_partly_cancelled = TRUE 
            THEN 1 
            ELSE 0 
        END AS cancellation_count,
        
        -- 4. (Optional) Keep the raw, unpenalized delay minutes just in case
        arrival_delay_min AS raw_arrival_delay_min

    FROM services_filled
""")

In [ ]:
if check_rowc:
    count_row_diff("services_filled", "services_penalty")

In [ ]:
if rstats:
    display(con.execute("SELECT * FROM services_penalty LIMIT 10").df())

Exclude all services with cancellations from the services dataset with the commented code below, but this ignores passengers perspective

In [ ]:
# con.execute("""
#     CREATE OR REPLACE VIEW services_penalty AS 
#     SELECT * FROM services_filled
#     WHERE 
#         -- Exclude cancelled arrivals
#         is_arrival_cancelled IS NOT TRUE
        
#         -- Exclude cancelled departures
#         AND is_departure_cancelled IS NOT TRUE
# """)

# # Quick verification: Check how many rows remain
# print(con.execute("SELECT count(*) FROM services_penalty").fetchone()[0])

In [ ]:
if rstats:
    # Checking arrival_delay_min distribution after penalty application
    check_arrival_delay_distribution("services_penalty")

## Transformation 1: adding 'to stations' columns to services dataset

Each service has atleast two rows: the departure (current Stop:Station, i.e. from station) and arrival (to station inferred from sorting the service line and time). 

source = departure station

target = arrival station

In [ ]:
# ==========================================
# STEP 2: Create Edges (Window Functions)
# ==========================================
# We materialize this as a TABLE to compute the expensive window functions once.
con.execute("""
    CREATE OR REPLACE VIEW services_with_edges AS 
    SELECT 
        *,
        -- From station is just the current row's station code
        station_code AS source,
            
        -- 1. Where are we going next? Get Destination (Next Row)
        LEAD(station_code) OVER (
            PARTITION BY service_id 
            ORDER BY COALESCE(departure_time, arrival_time) ASC
        ) AS target

    FROM services_penalty
""")
print("✅ Layer 2: Added Edge columns.")

✅ Layer 2: Added Edge columns.


Edges = 2931 (takes ~1min locally)

In [ ]:
if rstats:
    display(con.execute("""
        SELECT count(*) AS unique_edges
        FROM (
            SELECT DISTINCT source, target 
            FROM services_with_edges
            WHERE target IS NOT NULL
        )
    """).df())

## Transformation 2: Stations
We keep the stations dataset as ground truth for which stations exist (in the netherlands) and drop services with other stations

### A) Add station distances feature

In [ ]:
# ==========================================
# STEP 1: Flatten distance matrix dataset
# ==========================================
con.execute("""
    CREATE OR REPLACE VIEW station_distances_long AS 
    SELECT 
        Station AS source,          -- Rename 'Station' to 'source' for clarity
        target_station AS target, 
        CAST(distance AS INTEGER) AS distance
    FROM (
        UNPIVOT raw_station_distances
        ON COLUMNS(* EXCLUDE (Station))
        INTO
            NAME target_station
            VALUE distance
    )
    -- Filter 1: Explicitly remove the known garbage values
    WHERE distance NOT IN ('?', 'XXX')
    
    -- Filter 2: Ensure only valid numbers remain
    -- AND TRY_CAST(distance AS INTEGER) IS NOT NULL
""")

if rstats:
    # Let's verify the columns first to be safe:
    print("Unpivoted columns:", con.execute("DESCRIBE station_distances_long").df()['column_name'].tolist())

Following 2 stations from the distance connection matrix were not in stations

LEER
WR

In [ ]:
if rstats:
    con.execute("""
        -- 1. Get ALL stations from the matrix
        WITH matrix_stations AS (
            SELECT TRIM(source) AS station FROM station_distances_long
            UNION 
            SELECT TRIM(target) AS station FROM station_distances_long
        )
        
        -- 2. "Subtract" the official list
        SELECT station AS not_in_stations FROM matrix_stations
        
        EXCEPT 
        
        SELECT TRIM(code) FROM stations
    """).df()

In [ ]:
if rstats:
    con.execute("""
        SELECT TRIM(code) AS not_in_distances
        FROM stations
        
        EXCEPT 
        
        -- Subtract all stations found in the matrix
        (
            SELECT TRIM(source) FROM station_distances_long
            UNION 
            SELECT TRIM(target) FROM station_distances_long
        )
    """).df()

In [ ]:
# ==========================================
# STEP 2: Join Distances
# ==========================================
con.execute("""
    CREATE OR REPLACE VIEW services_edges_distances AS 
    SELECT 
        e.*, 
        d.distance AS distance
    FROM services_with_edges e
    LEFT JOIN station_distances_long d
        ON e.source = d.source 
        AND e.target = d.target
""")

print("✅ Layer 2A: Distances joined.")

✅ Layer 2A: Distances joined.


330 edges missing distances in services i.e. not in stations distances dataset ~(1 min)

These are REMOVED

In [ ]:
if rstats:
    # Check for "Orphan Edges" (Edges that exist but have no distance)
    orphan_edges = con.execute("""
        SELECT DISTINCT source, target 
        FROM services_edges_distances
        WHERE 
            distance IS NULL    -- Capture where the join failed
    """).df()

    if not orphan_edges.empty:
        print(f"⚠️ Warning: {len(orphan_edges)} routes are missing distance data.")
        display(orphan_edges.head())
    else:
        print("✅ Success: All edges have a corresponding distance.")

### B) Services with valid stations (i.e. occuring in 2023-09 stations rdt dataset)
REMOVED: We only consider the edges that are present in both distances matrix dataset and the stations dataset

In [ ]:
# ==========================================
# Filter Invalid Rows
# ==========================================
con.execute("""
    CREATE OR REPLACE VIEW services_valid AS 
    SELECT * FROM services_edges_distances
    WHERE 
        -- 1. Distance must be known (removes missing edges in distance from service data)
        distance IS NOT NULL
        
        -- 2. Source must be in the official station list
        AND (source IN (SELECT code FROM stations)
        
        -- 3. Target must be in the official station list
        AND target IN (SELECT code FROM stations))
""")

print("✅ Layer 2B: Filtered services to valid edges only.")

✅ Layer 2B: Filtered services to valid edges only.


In [ ]:
if check_rowc:
    count_row_diff("services_edges_distances", "services_valid")

Valid services ROWCOUNT = 
75_303_052 

In [ ]:
if rstats:
    check_arrival_delay_distribution("services_valid")

### (Old) Service delay bins (figures)

In [ ]:
# con.execute("""
#     CREATE OR REPLACE VIEW delay_bins AS
#     SELECT 
#         -- Create the bins on the fly (or use your view)
#         CASE 
#             WHEN arrival_delay_min <= 0 THEN 'On Time (inf, 0]'
#             WHEN arrival_delay_min < 30 THEN 'Small Delay (0, 30)'
#             WHEN arrival_delay_min < 60 THEN 'Medium Delay [30, 60)'
#             ELSE 'Large Delay [60, inf)'
#         END AS delay_category,
        
#         -- Helper column for sorting the chart correctly
#         CASE 
#             WHEN arrival_delay_min <= 0 THEN 1
#             WHEN arrival_delay_min < 30 THEN 2
#             WHEN arrival_delay_min < 60 THEN 3
#             ELSE 4
#         END AS sort_order,

#         -- Count services in each bin
#         COUNT(*) as count
#     FROM services_valid
#     WHERE arrival_delay_min IS NOT NULL
#     GROUP BY 1, 2
#     ORDER BY sort_order
# """)

In [42]:
# binned_data_output_path = filtered_output_root_folder / "full_services_dataset/service_delay_bins.parquet"

In [43]:
# con.execute(f"COPY delay_bins TO '{binned_data_output_path}' (FORMAT PARQUET)")
# print("Done! Saved as ", binned_data_output_path)

In [ ]:
# import matplotlib.pyplot as plt
# import pandas as pd

# # 1. Load Data
# binned_data_file = filtered_output_root_folder / "full_services_dataset/delay_bins.parquet"
# df_bins = pd.read_parquet(binned_data_file)

# # 2. Calculate Totals (Needed for %)
# total_services = df_bins['count'].sum()
# on_time_count = df_bins[df_bins['sort_order'] == 1]['count'].sum()
# delayed_count = df_bins[df_bins['sort_order'] > 1]['count'].sum()

# # 3. Setup the Plot Area
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# # --- Plot 1: Multi-Class Bar Chart ---
# bars = ax1.bar(
#     df_bins['delay_category'], 
#     df_bins['count'], 
#     color=['#2ca02c', '#ff7f0e', '#d62728', '#8c564b'], 
#     edgecolor='black', 
#     alpha=0.8
# )

# ax1.set_title('Distribution of Services Arrival Delays (Categorical)', fontsize=14)
# ax1.set_ylabel('Number of Services', fontsize=12)
# ax1.grid(axis='y', linestyle='--', alpha=0.3)

# # Add value labels AND percentages on top of bars
# for bar in bars:
#     height = bar.get_height()
#     percentage = (height / total_services) * 100  # Calculate %
    
#     # Format: "1,234 \n (12.5%)"
#     label_text = f'{int(height):,}\n({percentage:.1f}%)'
    
#     ax1.text(bar.get_x() + bar.get_width()/2., height,
#              label_text,
#              ha='center', va='bottom', fontsize=10, fontweight='bold')

# # --- Plot 2: Binary Pie Chart ---
# labels = [f'On Time\n({on_time_count:,})', f'Delayed\n({delayed_count:,})']
# sizes = [on_time_count, delayed_count]
# colors = ['#2ca02c', '#d62728'] 
# explode = (0, 0.1) 

# ax2.pie(sizes, explode=explode, labels=labels, colors=colors,
#         autopct='%1.1f%%', shadow=True, startangle=140, textprops={'fontsize': 11})
# ax2.set_title('Binary Classification: Services On Time vs Delayed', fontsize=14)

# # 4. Save and Show
# plt.tight_layout()
# # plt.savefig(f'{media_folder}/delay_classification_plots.png')
# # print("✅ Plots saved as 'delay_classification_plots.png'")

## Transformation 3: Monthly Aggregation of Trajectories 

We aggregate all service data to the service yearmonth. Then we add a  

We add binary classification column of is significantly delayed utilizing the median ratio delayed trajectory/total trajectories of total of the dataset

(Merel's and Jonathan paper utilized the COUNT of services on a trajectory for their classification label?) -> Lei's original work was based on edge removal classification based on actual edges that were present at month M but not M+1 as ground truth

Delay classification is performance based (Passenger experience): under these topological conditions will a train on a trajectory to what extent will the train be delayed?
 
Lei's: Edge Removal, infrastructural failure "Under these conditions, does the connection physically exist?"


In [45]:
# # Inspecting columns to keep/drop for graph features
# display(con.execute("""
#     SELECT column_name 
#     FROM (DESCRIBE services_valid)
# """).df())

### Create clean monthly services dataset (run once)

DOWNLOAD: downloads/data/filtered/final_aggregated/services_monthly.parquet

UNCOMMENT

In [46]:
# if snellius is True:
# 	con.execute("""
# 		CREATE OR REPLACE TABLE services_monthly_agg AS
# 		SELECT
# 			-- grouping keys
# 			source,
# 			target,
# 			strftime(service_date, '%Y-%m') AS YearMonth,
# 			EXTRACT(year FROM service_date) AS year,
# 			EXTRACT(month FROM service_date) AS month,

# 			-- static features
# 			FIRST(distance) AS distance,

# 			-- departure/arrival stats aggregated for month
# 			SUM(departure_delay_min) AS total_departure_delay_minutes,
# 			ROUND(AVG(departure_delay_min), 2) AS avg_departure_delay_minutes,
# 			COUNT(*) FILTER (WHERE departure_delay_min > 0) AS count_delayed_departure_services,

# 			SUM(arrival_delay_min) AS total_delay_minutes,
# 			ROUND(AVG(arrival_delay_min), 2) AS avg_delay_minutes,
# 			COUNT(*) FILTER (WHERE arrival_delay_min > 0) AS count_delayed_services,

# 			COUNT(*) AS total_services,
# 			ROUND(COUNT(*) FILTER (WHERE departure_delay_min > 0) * 1.0 / NULLIF(COUNT(*),0),4) AS ratio_departure_delayed,
# 			ROUND(COUNT(*) FILTER (WHERE arrival_delay_min > 0) * 1.0 / NULLIF(COUNT(*),0),4) AS ratio_arrival_delayed,

# 			COUNT(*) FILTER (WHERE has_platform_change IS TRUE) AS count_platform_changes,
# 			histogram(train_type) AS train_type_counts

# 		FROM services_valid
# 		GROUP BY 1, 2, 3, 4, 5
# 		-- Filter out any edge/month combination with fewer than 4 total services
# 		HAVING COUNT(*) >= 4
# 	""")
# 	print("✅ Aggregated table 'services_monthly_agg' created.")

In [ ]:
if snellius is True:
    con.execute("""
        CREATE OR REPLACE TABLE services_monthly_agg AS
        SELECT
            -- grouping keys
            source,
            target,
            strftime(service_date, '%Y-%m') AS YearMonth,
            EXTRACT(year FROM service_date) AS year,
            EXTRACT(month FROM service_date) AS month,

            -- static features
            FIRST(distance) AS distance,

            -- departure/arrival stats aggregated for month
            SUM(departure_delay_min) AS total_departure_delay_minutes,
            ROUND(AVG(departure_delay_min), 2) AS avg_departure_delay_minutes,
            COUNT(*) FILTER (WHERE departure_delay_min > 0) AS count_delayed_departure_services,

            SUM(arrival_delay_min) AS total_delay_minutes,
            ROUND(AVG(arrival_delay_min), 2) AS avg_delay_minutes,
            COUNT(*) FILTER (WHERE arrival_delay_min > 0) AS count_delayed_services,

            -- 1. Total Services
            COUNT(*) AS total_services,
            SUM(total_service_count) AS monthly_true_total_services, -- This should be the same as total_services, but we can verify
            
            -- 2. Pure Arrival Delays
            SUM(pure_delay_count) AS monthly_pure_delay_count,

            -- 3. Cancellation Count
            SUM(cancellation_count) AS monthly_cancellation_count,

            ROUND(COUNT(*) FILTER (WHERE departure_delay_min > 0) * 1.0 / NULLIF(COUNT(*),0),4) AS ratio_departure_delayed,
            ROUND(COUNT(*) FILTER (WHERE arrival_delay_min > 0) * 1.0 / NULLIF(COUNT(*),0),4) AS ratio_arrival_delayed,

            COUNT(*) FILTER (WHERE has_platform_change IS TRUE) AS count_platform_changes,
            histogram(train_type) AS train_type_counts

        FROM services_valid
        GROUP BY 1, 2, 3, 4, 5
        -- Filter out any edge/month combination with fewer than 4 total services
        HAVING COUNT(*) >= 4
    """)
    print("✅ Aggregated table 'services_monthly_agg' created.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Aggregated table 'services_monthly_agg' created.


# Feature Engineering


## Operational edge service feature

time decayed weight feature (based on total service lags), inspired from brent's operational lag features and highlighting lei's weight importance and merel data leakage problem
12months lookback window due to time schedule announced annually in advance

$$W_{decayed} = \frac{V_{t-1}}{\Delta m} \quad \text{for } 1 \le \Delta m \le 12$$

$W_{decayed}$: The Decayed Edge Weight, representing the effective "strength" or "importance" of the trajectory in the current month's graph topology.

$V_{t-1}$: The Lagged Service Volume (total_services_lag1), representing the number of services completed during the most recent active month within the look-back window.

$\Delta m$: The Temporal Lag (months_since_last_service), representing the number of months elapsed between the current month $t$ and the most recent active month $t_{last}$.

last_active_month = None for cold start case or more than 12 months ago (indicating new route or out of annual range)

In [ ]:
if snellius is True:
    con.execute("""
        CREATE OR REPLACE TABLE services_monthly_agg AS
        WITH last_active_lookup AS (
            SELECT 
                *,
                -- 1. Most recent VOLUME (Windowed MAX handles gaps, COALESCE handles the first month)
                COALESCE(
                    MAX(CASE WHEN total_services > 0 THEN total_services ELSE 0 END) OVER (
                        PARTITION BY source, target
                        ORDER BY YearMonth
                        ROWS BETWEEN 12 PRECEDING AND 1 PRECEDING
                    ), 
                0) AS last_active_volume,

                -- 2. Most recent DATE (For distance calculation)
                MAX(CASE WHEN total_services > 0 THEN YearMonth END) OVER (
                    PARTITION BY source, target
                    ORDER BY YearMonth
                    ROWS BETWEEN 12 PRECEDING AND 1 PRECEDING
                ) AS last_active_month
            FROM services_monthly_agg
        ),
        staleness_calc AS (
            SELECT 
                *,
                -- 3. Calculate distance (Impute to 12 if never active)
                COALESCE(
                    date_diff('month', 
                        CAST(last_active_month || '-01' AS DATE), 
                        CAST(YearMonth || '-01' AS DATE)
                    ), 12
                ) AS months_since_last_service
            FROM last_active_lookup
        )
        -- FINAL STEP: Combine Volume and Distance into the Edge Weight
        SELECT 
            *,
            -- 4. NEW Decayed Weight with NULL protection
            ROUND(
                COALESCE(last_active_volume, 0) * (1.0 / NULLIF(months_since_last_service, 0)), 
            2) AS decayed_edge_weight,
            
            CASE WHEN last_active_month IS NULL THEN 1 ELSE 0 END AS is_new_or_stale_route

        FROM staleness_calc;
    """)

In [ ]:
if cdata_new:
    aggregated_path = filtered_output_root_folder / "final_aggregated/services_monthly.parquet"
    con.execute(f"COPY services_monthly_agg TO '{aggregated_path}' (FORMAT PARQUET)")
    print("Done! Saved as ", aggregated_path)

Done! Saved as  /home/jbao/NS_Thesis/downloads/data/filtered/final_aggregated/services_monthly.parquet


## Network Topology (Graph) Features 

### Calculate graph features
(Brent's Disruptions adaptation & optimized)

Distinguishment from Kämper: dynamic (edges) topology feature creation, lagged time decayed weight utilization. Recency bias due to slow changing nature of the network

Distinguishment from Brent's:
Graph structure of Stations (nodes) remained consistent for all graphs in prior work (brent), as he utilized the stations of the whole dataset to create the daily graphs (this result in including stations that are not connected, so he only alternated the graph structure in edges i.e. the connectivity).

-> we only create a graph that is connected (monthly stations), i.e. total_services is >= 1, which exclude stations that are do not have service in that hour from the graph (i.e. they do not exist as nodes) essentially alternating the graph structure (both nodes and edges connectivity). 

Compared 

We also return None for hourly 

Feature we utilize now: source, target, 

In [ ]:
if create_graph_features:
    import networkx as nx

    # 2. Get all valid YearMonth combinations first
    print("Fetching monthly time slots...")
    time_slots = con.execute(f"""
        SELECT DISTINCT YearMonth
        FROM services_monthly_agg
        ORDER BY 1
    """).fetchall()

    print(f"✅ Found {len(time_slots)} monthly slots to process.")

    # 3. Initialize Storage
    # Key will be string: '2023-01', '2023-02', etc.
    monthly_node_features = {}
    monthly_edge_features = {}

    # 4. Helper: Build Graph for One Month
    def get_monthly_graph(year_month_val):
        query = f"""
            SELECT 
                source, 
                target,
                distance,
                decayed_edge_weight AS weight,
            FROM services_monthly_agg
            WHERE YearMonth = '{year_month_val}'
        """
        df = con.execute(query).df()
        
        if df.empty:
            return None

        # Fast Bulk Loading
        G = nx.from_pandas_edgelist(
            df, 
            source='source', 
            target='target', 
            edge_attr=['weight', 'distance'],
            create_using=nx.DiGraph()
        )
        return G

    # 5. Helper: Extract Graph Features
    def extract_graph_features(G):
        # --- Node Features (Directed) ---
        n_feats = {}
        for node in G.nodes():
            deg = G.degree(node)
            
            # Weighted degree (sum of rides planned), Brent's weighted_degree to neighbours
            w_deg_outbound = G.out_degree(node, weight='weight')

            w_deg_inbound = G.in_degree(node, weight='weight')			

            # 1. Avg Distance of trains LEAVING here (Out-degree), Brent's avg_distance to neighbours
            out_edges = list(G.out_edges(node, data='distance'))
            avg_dist_out = sum(d for _, _, d in out_edges) / len(out_edges) if out_edges else 0

            # 2. Avg Distance of trains ARRIVING here (In-degree)
            in_edges = list(G.in_edges(node, data='distance'))
            avg_dist_in = sum(d for _, _, d in in_edges) / len(in_edges) if in_edges else 0
            
            n_feats[node] = {
                'degree': deg,
                'w_deg_inbound': w_deg_inbound,
                'w_deg_outbound': w_deg_outbound,
                'w_deg_total': w_deg_inbound + w_deg_outbound,
                'avg_distance_inbound': avg_dist_in,
                'avg_distance_outbound': avg_dist_out,
                'avg_distance_total': avg_dist_in + avg_dist_out  # Total avg distance (In + Out)
            }

        # --- Edge Features (Undirected for Topology Metrics) ---
        # We create the undirected view ONCE per graph, not per edge (optimization)
        G_undir = G.to_undirected()
        
        e_feats = {}
        for u, v, data in G.edges(data=True):
            # Common Neighbors
            cn = list(nx.common_neighbors(G_undir, u, v))
            
            # Calculate complex metrics safely
            try:
                # Note: These return iterators, so we list() them and grab the score
                jaccard = list(nx.jaccard_coefficient(G_undir, [(u, v)]))[0][2]
                adamic = list(nx.adamic_adar_index(G_undir, [(u, v)]))[0][2]
                res_alloc = list(nx.resource_allocation_index(G_undir, [(u, v)]))[0][2]
            except:
                jaccard, adamic, res_alloc = 0, 0, 0

            e_feats[(u, v)] = {
                'common_neighbors': len(cn),
                'jaccard_coefficient': jaccard,
                'preferential_attachment': G.degree(u) * G.degree(v),
                'adamic_adar_index': adamic,
                'resource_allocation_index': res_alloc,
                'weight': data['weight'],
                'distance': data['distance'],
            }
            
        return n_feats, e_feats

Fetching monthly time slots...
✅ Found 72 monthly slots to process.


### Create monthly graph features
DOWNLOAD: (folder) downloads/data/filtered/graph_features_output/

In [ ]:
if create_graph_features:

    # Configuration
    CHUNK_SIZE = 12 # Save a checkpoint every 12 months

    # Buffers (Temporary Lists)
    node_buffer = []
    edge_buffer = []
    batch_count = 0

    print(f" Processing {len(time_slots)} months. Saving to '{GRAPH_OUTPUT_DIR}'...")

    # --- MAIN LOOP ---
    # time_slots from fetchall() looks like: [('2023-01',), ('2023-02',)]
    for i, (year_month_val, ) in enumerate(time_slots):
        
        # A. Build Graph
        G = get_monthly_graph(year_month_val)
        if G is None: continue 

        # B. Extract Features
        n_feats, e_feats = extract_graph_features(G)
        
        # C. Flatten & Buffer (Convert Dict -> List of Rows)
        # Process Node Features
        for node, feats in n_feats.items():
            node_buffer.append({
                'YearMonth': year_month_val,
                'station': node,
                **feats  # Unpacks {'degree': 10, ...}
            })

        # Process Edge Features
        for (u, v), feats in e_feats.items():
            edge_buffer.append({
                'YearMonth': year_month_val,
                'source': u,
                'target': v,
                **feats # Unpacks {'jaccard': 0.5, ...}
            })
            
        # D. Save Checkpoint (Batch Flush)
        if len(node_buffer) > 0 and (i + 1) % CHUNK_SIZE == 0:
            # Save Nodes
            pd.DataFrame(node_buffer).to_parquet(
                f"{GRAPH_OUTPUT_DIR}/nodes_part_{batch_count}.parquet", index=False
            )
            # Save Edges
            pd.DataFrame(edge_buffer).to_parquet(
                f"{GRAPH_OUTPUT_DIR}/edges_part_{batch_count}.parquet", index=False
            )
            
            # Clear Memory
            node_buffer = []
            edge_buffer = []
            batch_count += 1

    # E. Final Flush (Save whatever is left)
    if node_buffer:
        pd.DataFrame(node_buffer).to_parquet(f"{GRAPH_OUTPUT_DIR}/nodes_part_{batch_count}.parquet", index=False)
        pd.DataFrame(edge_buffer).to_parquet(f"{GRAPH_OUTPUT_DIR}/edges_part_{batch_count}.parquet", index=False)

    print("✅ Done! Data saved to parquet files.")

 Processing 72 months. Saving to '/home/jbao/NS_Thesis/downloads/data/filtered/graph_features_output'...


✅ Done! Data saved to parquet files.


### Join graph features with services (run once)

We read this back in as aggregated_hourly_graph_features in duckdb

DOWNLOAD: downloads/data/filtered/final_aggregated/services_hourly_graph_features.parquet

We merge by the following:
Join NODES (graph_nodes) on 
hour = arrival_hour
date = service_date
station = source

EDGES (graph_edges) on
hour = arrival_hour
date = service_date
source = source
target = target
drop distance
drop weight

In [52]:
graph_nodes_path = str(GRAPH_OUTPUT_DIR / "nodes_part_*.parquet")
graph_edges_path = str(GRAPH_OUTPUT_DIR / "edges_part_*.parquet")

# Load into DuckDB Views
con.execute(f"CREATE OR REPLACE VIEW graph_nodes AS SELECT * FROM read_parquet('{graph_nodes_path}')")
con.execute(f"CREATE OR REPLACE VIEW graph_edges AS SELECT * FROM read_parquet('{graph_edges_path}')")

# Sanity Check: Verify we actually found files and loaded rows
print("✅Done loading graph features into duckdb!")

✅Done loading graph features into duckdb!


In [ ]:
con.execute("""
    CREATE OR REPLACE VIEW services_monthly_graph_features AS
    SELECT
        base.*,
        
        -- Source Node Features
        src.degree                AS src_degree,
        src.w_deg_inbound         AS src_w_deg_inbound,
        src.w_deg_outbound        AS src_w_deg_outbound,
        src.w_deg_total 		  AS src_w_deg_total,
        src.avg_distance_total    AS src_avg_distance_total,
        src.avg_distance_inbound  AS src_avg_distance_in,
        src.avg_distance_outbound AS src_avg_distance_out,
        
        -- Target Node Features
        tgt.degree                AS tgt_degree,
        tgt.avg_distance_total    AS tgt_avg_distance_total,
        tgt.avg_distance_inbound  AS tgt_avg_distance_in,
        tgt.avg_distance_outbound AS tgt_avg_distance_out,
        tgt.w_deg_inbound         AS tgt_w_deg_inbound,
        tgt.w_deg_outbound        AS tgt_w_deg_outbound,
        tgt.w_deg_total 		  AS tgt_w_deg_total,
        
        -- Edge Features
        e.common_neighbors,
        e.jaccard_coefficient,
        e.preferential_attachment,
        e.adamic_adar_index,
        e.resource_allocation_index
        
    FROM services_monthly_agg base
    
    LEFT JOIN graph_nodes src
        ON base.YearMonth = src.YearMonth
        AND base.source = src.station
        
    LEFT JOIN graph_nodes tgt
        ON base.YearMonth = tgt.YearMonth
        AND base.target = tgt.station
        
    LEFT JOIN graph_edges e
        ON base.YearMonth = e.YearMonth
        AND base.source = e.source
        AND base.target = e.target
""")
print("✅ View 'services_monthly_graph_features' created successfully.")

# Save as parquet file 
if cdata_new:
    graph_features_output_path = filtered_output_root_folder / "final_aggregated/services_monthly_graph_features.parquet"
    con.execute(f"COPY services_monthly_graph_features TO '{graph_features_output_path}' (FORMAT PARQUET)")
    print("Done! Saved as ", graph_features_output_path)

✅ View 'services_monthly_graph_features' created successfully.


Done! Saved as  /home/jbao/NS_Thesis/downloads/data/filtered/final_aggregated/services_monthly_graph_features.parquet


In [ ]:
# 1. Month-by-Month Edge Counts
monthly_edges_query = """
    SELECT 
        YearMonth, 
        COUNT(*) AS total_active_edges
    FROM services_monthly_graph_features
    GROUP BY YearMonth
    ORDER BY YearMonth
"""
print("--- ACTIVE EDGES PER MONTH ---")
display(con.execute(monthly_edges_query).df())
print("\n")


# 2. Overall Statistics for the Network's Topology
edge_stats_query = """
    WITH monthly_counts AS (
        SELECT 
            YearMonth, 
            COUNT(*) AS num_edges
        FROM services_monthly_graph_features
        GROUP BY YearMonth
    )
    SELECT 
        MIN(num_edges) AS min_edges_in_a_month,
        MAX(num_edges) AS max_edges_in_a_month,
        ROUND(AVG(num_edges), 0) AS avg_edges_per_month,
        ROUND(STDDEV(num_edges), 0) AS stddev_edges
    FROM monthly_counts
"""
print("--- OVERALL EDGE STATISTICS ---")
edge_df = con.execute(edge_stats_query).df()

--- ACTIVE EDGES PER MONTH ---


,YearMonth,total_active_edges
0,2019-01,685
1,2019-02,664
2,2019-03,669
3,2019-04,682
4,2019-05,680
...,...,...
67,2024-08,717
68,2024-09,723
69,2024-10,714
70,2024-11,724




--- OVERALL EDGE STATISTICS ---


## Weather features
Since weather is quite chaotic, the average is an appropriate estimate of a month (as the exact measures would be unknown and require to be predicted).

In [ ]:
con.execute("""
    CREATE OR REPLACE VIEW monthly_weather_features AS
    SELECT 
        station_code,
        strftime(weather_date, '%Y-%m') AS YearMonth,
        
        -- Strict Monthly Averages
        AVG(temperature_2m) AS avg_temperature_2m,
        SUM(rain) AS avg_rain,
        SUM(snowfall) AS avg_snowfall,
        AVG(snow_depth) AS avg_snow_depth,
        AVG(wind_speed_10m) AS avg_wind_speed_10m,
        MAX(wind_gusts_10m) AS avg_wind_gusts_10m,
        AVG(soil_temperature_0_to_7cm) AS avg_soil_temperature_0_to_7cm
        
    FROM hourly_weather_features
    GROUP BY 
        station_code, 
        YearMonth
""")

print("✅ View 'monthly_weather_features' created with strict averages.")

✅ View 'monthly_weather_features' created with strict averages.


In [ ]:
con.execute("""
    CREATE OR REPLACE VIEW services_weather_graph_features_merged AS
    SELECT 
        base.*,
        
        -- Source Station Weather
        src_w.avg_temperature_2m          AS src_avg_temperature_2m,
        src_w.avg_rain                    AS src_avg_rain,
        src_w.avg_snowfall                AS src_avg_snowfall,
        src_w.avg_snow_depth              AS src_avg_snow_depth,
        src_w.avg_wind_speed_10m          AS src_avg_wind_speed_10m,
        src_w.avg_wind_gusts_10m          AS src_avg_wind_gusts_10m,
        src_w.avg_soil_temperature_0_to_7cm AS src_avg_soil_temperature,
        
        -- Target Station Weather
        tgt_w.avg_temperature_2m          AS tgt_avg_temperature_2m,
        tgt_w.avg_rain                    AS tgt_avg_rain,
        tgt_w.avg_snowfall                AS tgt_avg_snowfall,
        tgt_w.avg_snow_depth              AS tgt_avg_snow_depth,
        tgt_w.avg_wind_speed_10m          AS tgt_avg_wind_speed_10m,
        tgt_w.avg_wind_gusts_10m          AS tgt_avg_wind_gusts_10m,
        tgt_w.avg_soil_temperature_0_to_7cm AS tgt_avg_soil_temperature
        
    FROM services_monthly_graph_features base
    
    -- 1. Attach weather for the SOURCE station
    LEFT JOIN monthly_weather_features src_w
        ON base.YearMonth = src_w.YearMonth
        AND base.source = src_w.station_code
        
    -- 2. Attach weather for the TARGET station
    LEFT JOIN monthly_weather_features tgt_w
        ON base.YearMonth = tgt_w.YearMonth
        AND base.target = tgt_w.station_code
""")

print("✅ View 'services_weather_graph_features_merged' created.")
print("   - Source and Target weather features successfully attached.")

✅ View 'services_weather_graph_features_merged' created.
   - Source and Target weather features successfully attached.


### NOTE HOURLY WEATHER DATA for HOURLY AGGREGATIONS
(Commented)
 The hourly weather dataset has some duplicate entries. These duplicate entries are present at hour 2:00 for only certain dates, around the same dates annually, for all locations to be affected.
Upon further investigation and giving it some thought, it was noted that this was likely caused by daylight saving time.

If you opt to use an hourly dataset; you could for example approach this in several ways:
- You could naively assume you do not know which of both entries represents 2:00 or 3:00 actual values. So you take the average of both to be safe and impute it to both hours. 
- You could perform interpolation between for example 1:00 - 4:00 to determine which is which.

In [57]:
# # Check for duplicates in the weather table on join keys
# check_weather = con.execute("""
#     SELECT weather_date, weather_hour, station_code, COUNT(*)
#     FROM hourly_weather_features
#     GROUP BY weather_date, weather_hour, station_code
#     HAVING COUNT(*) > 1
# """).df()

In [58]:
# check_weather['weather_hour'].unique()

In [59]:
# check_weather['weather_date'].unique()

In [ ]:
# # 1. Get the list of duplicate keys first (as you did)
# # We use a CTE or subquery to isolate the keys
# query_duplicates = """
#     WITH duplicates AS (
#         SELECT weather_date, weather_hour, station_code
#         FROM hourly_weather_features
#         GROUP BY weather_date, weather_hour, station_code
#         HAVING COUNT(*) > 1
#     )
    
#     -- 2. Select ALL columns from the main table where the key matches a duplicate
#     SELECT t.*
#     FROM hourly_weather_features t
#     JOIN duplicates d
#         ON t.weather_date = d.weather_date
#         AND t.weather_hour = d.weather_hour
#         AND t.station_code = d.station_code
#     ORDER BY t.station_code, t.weather_date, t.weather_hour
# """

# # Display the actual conflicting values
# df_duplicate_values = con.execute(query_duplicates).df()
# display(df_duplicate_values)

In [61]:
# df_duplicate_values[df_duplicate_values.isna().any(axis=1)]

In [ ]:
# # Create a clean, unique weather view
# con.execute("""
#     CREATE OR REPLACE VIEW v_weather_hourly_clean AS
#     SELECT 
#         -- Grouping Keys (The unique identifier)
#         weather_date,
#         weather_hour,
#         station_code,
        
#         -- Aggregate the features to handle variations
#         AVG(temperature_2m) as temperature_2m,
#         AVG(rain) as rain,
#         AVG(snowfall) as snowfall,
#         AVG(snow_depth) as snow_depth,
#         AVG(wind_speed_10m) as wind_speed_10m,
#         AVG(wind_gusts_10m) as wind_gusts_10m,
#         AVG(soil_temperature_0_to_7cm) as soil_temperature_0_to_7cm
        
#     FROM hourly_weather_features
#     GROUP BY weather_date, weather_hour, station_code
# """)

# print("✅ Created 'v_weather_hourly_clean' with duplicates merged via Average.")

In [ ]:
# # 2. First initial join Weather to the Main Dataset, resulted in additional rows?
# con.execute("""
#     CREATE OR REPLACE VIEW services_weather_graph_features_merged AS
#     SELECT 
#         -- 1. All existing Service & Graph columns
#         base.*,
        
#         -- 2. Weather Features (prefixed for clarity)
#         w.temperature_2m,
#         w.rain,
#         w.snowfall,
#         w.snow_depth,
#         w.wind_speed_10m,
#         w.wind_gusts_10m,
#         w.soil_temperature_0_to_7cm
        
#     FROM aggregated_hourly_graph_features base
    
#     -- Join Weather on Source Station
#     LEFT JOIN v_weather_hourly_clean w
#         ON base.service_date = w.weather_date
#         AND base.earliest_departure_hour = w.weather_hour
#         AND base.source = w.station_code
# """)

# print("✅ View 'services_weather_graph_features_merged' created.")

In [64]:
# # Check which hours have the most missing weather data
# con.execute("""
#     SELECT 
#         arrival_hour,
#         COUNT(*) as total_rows,
#         COUNT(temperature_2m) as rows_with_weather,
#         (COUNT(*) - COUNT(temperature_2m)) as missing_matches
#     FROM services_weather_graph_features_merged
#     GROUP BY arrival_hour
#     ORDER BY missing_matches DESC
# """).df()

In [ ]:
# # 1. Inspect the content of the duplicates
# query_inspection = """
#     WITH duplicates_list AS (
#         -- Find the (date, hour, station) keys that have > 1 row
#         SELECT weather_date, weather_hour, station_code
#         FROM hourly_weather_features
#         GROUP BY 1, 2, 3
#         HAVING COUNT(*) > 1
#     )
    
#     SELECT 
#         t.station_code,
#         t.weather_date,
#         t.weather_hour,
#         -- Select the specific value columns to compare
#         t.temperature_2m,
#         t.rain,
#         t.wind_speed_10m,
#         t.snowfall
#     FROM hourly_weather_features t
#     INNER JOIN duplicates_list d
#         ON t.weather_date = d.weather_date
#         AND t.weather_hour = d.weather_hour
#         AND t.station_code = d.station_code
#     ORDER BY t.station_code, t.weather_date, t.weather_hour
# """

# # Display the first 20 rows to see the pattern
# df_inspection = con.execute(query_inspection).df()
# display(df_inspection.head(20))

## Calendar features

In [ ]:
con.execute("""
    CREATE OR REPLACE VIEW services_with_holiday AS
    
    -- 1. Pre-aggregate the holidays into monthly buckets first
    WITH monthly_holidays AS (
        SELECT 
            strftime(date, '%Y-%m') AS YearMonth,
            COUNT(*) AS total_holidays
        FROM raw_holidays
        GROUP BY 1
    )
    
    SELECT 
        -- 2. Keep all existing columns
        base.*,
        
        -- 3. Binary into monthly count
        COALESCE(h.total_holidays, 0) AS total_holidays
        
    FROM services_weather_graph_features_merged base
    
    -- 4. Left Join on the monthly string instead of the exact date
    LEFT JOIN monthly_holidays h
        ON base.YearMonth = h.YearMonth
""")

print("✅ Added 'total_holidays' column (monthly count).")

✅ Added 'total_holidays' column (monthly count).


In [ ]:
con.execute("""
    CREATE OR REPLACE VIEW services_with_holiday AS
    
    -- 1. Create dynamic start and end dates based on your actual data
    WITH date_bounds AS (
        SELECT 
            CAST(MIN(YearMonth) || '-01' AS DATE) AS min_date,
            LAST_DAY(CAST(MAX(YearMonth) || '-01' AS DATE)) AS max_date
        FROM services_weather_graph_features_merged
    ),
    
    -- 2. Generate a daily calendar for that exact timeframe
    calendar AS (
        SELECT unnest(generate_series(min_date, max_date, INTERVAL 1 DAY)) AS cal_date
        FROM date_bounds
    ),
    
    -- 3. Calculate exact days and weekends per month
    monthly_calendar_stats AS (
        SELECT 
            strftime(cal_date, '%Y-%m') AS YearMonth,
            COUNT(*) AS total_days,
            -- DuckDB dayofweek: 0 is Sunday, 6 is Saturday
            SUM(CASE WHEN dayofweek(cal_date) IN (0, 6) THEN 1 ELSE 0 END) AS weekend_days
        FROM calendar
        GROUP BY 1
    ),
    
    -- 4. Calculate holidays (separating out the weekday ones)
    monthly_holidays AS (
        SELECT 
            strftime(date, '%Y-%m') AS YearMonth,
            COUNT(*) AS total_holidays,
            -- Only count holidays that fall on Mon-Fri (1 through 5)
            SUM(CASE WHEN dayofweek(date) NOT IN (0, 6) THEN 1 ELSE 0 END) AS weekday_holidays
        FROM raw_holidays
        GROUP BY 1
    )
    
    -- 5. Merge everything together and calculate the final ratios
    SELECT 
        base.*,
        
        -- The Raw Count
        COALESCE(h.total_holidays, 0) AS total_holidays,
        
        -- The Density Ratio (Total Holidays / Days in Month)
        ROUND(COALESCE(h.total_holidays, 0) * 1.0 / c.total_days, 4) AS holiday_density_ratio,
        
        -- The Weekend Days
        c.weekend_days,
        
        -- The Working Days (Total Days - Weekends - Weekday Holidays)
        (c.total_days - c.weekend_days - COALESCE(h.weekday_holidays, 0)) AS working_days
        
    FROM services_weather_graph_features_merged base
    
    -- Attach the calendar math
    LEFT JOIN monthly_calendar_stats c
        ON base.YearMonth = c.YearMonth
        
    -- Attach the holiday math
    LEFT JOIN monthly_holidays h
        ON base.YearMonth = h.YearMonth
""")

print("✅ Added 'total_holidays', 'holiday_density_ratio', 'weekend_days', and 'working_days' features.")

✅ Added 'total_holidays', 'holiday_density_ratio', 'weekend_days', and 'working_days' features.


## Target Label Class
Is significantly delayed binary target classification (median of arrival delay of the whole dataset as classification target) 

Global Arrival Delay Ratio Median Threshold = 26,29%

In [ ]:
# con.execute("""
#     CREATE OR REPLACE VIEW services_labeled AS
    
#     -- 1. Calculate the exact median across all months and edges
#     WITH global_stats AS (
#         SELECT median(ratio_arrival_delayed) AS global_median
#         FROM services_with_holiday
#         WHERE ratio_arrival_delayed IS NOT NULL
#     )
    
#     -- 2. Apply the threshold to create the binary label
#     SELECT 
#         base.*,
        
#         -- Expose the median just in case you want to review it later
#         g.global_median,
        
#         -- 1 = "Bad" (Worse than median delays)
#         -- 0 = "Good" (Better than or equal to median delays)
#         CASE 
#             WHEN base.ratio_arrival_delayed > g.global_median THEN 1
#             ELSE 0
#         END AS target_binary
        
#     FROM services_with_holiday base
#     CROSS JOIN global_stats g
# """)

# print("✅ View 'services_labeled' created.")
# print("   - Binary classification target 'target_binary' added via global median split.")

#### Global Median target + sensitivity analysis with alternative thresholds (Absolute and month relative)

In [ ]:
con.execute("""
    CREATE OR REPLACE VIEW services_labeled AS
    
    -- 1. Calculate the exact median across all data
    WITH global_stats AS (
        SELECT median(ratio_arrival_delayed) AS global_median
        FROM services_with_holiday
        WHERE ratio_arrival_delayed IS NOT NULL
    )
    
    -- 2. Build the final view with ALL targets
    SELECT 
        base.*,
        
        -- Expose the global median just in case you want to review it later
        g.global_median,
        
        -- TARGET 1: The Original Global Baseline, Is Significantly Delayed? (global median)
        CASE 
            WHEN base.ratio_arrival_delayed > g.global_median THEN 1
            ELSE 0
        END AS target_binary,
        
        -- TARGET 2: Absolute Threshold (i.e. at least 20 percent of the services is delayed)
        CASE 
            WHEN base.ratio_arrival_delayed > 0.20 THEN 1 
            ELSE 0 
        END AS target_absolute,
        
        -- SAVE THE THRESHOLD FOR EDA
        median(base.ratio_arrival_delayed) OVER (
            PARTITION BY base.month
        ) AS target_threshold_month_relative,
            
        -- TARGET 3: Month-Relative Threshold (Aggregated 1-12 across all years)
        CASE 
            WHEN base.ratio_arrival_delayed > median(base.ratio_arrival_delayed) OVER (
                PARTITION BY base.month
            ) THEN 1 
            ELSE 0 
        END AS target_relative
        
    FROM services_with_holiday base
    CROSS JOIN global_stats g
    WHERE base.ratio_arrival_delayed IS NOT NULL
""")

print("✅ View 'services_labeled' created with all target definitions.")

✅ View 'services_labeled' created with all target definitions.


In [ ]:
check_target_query = """
    SELECT 
        -- Since it's the same on every row, MAX just grabs that single value
        MAX(global_median) AS exact_global_median,
        
        -- Count how many rows became 1s and 0s
        COUNT(*) FILTER (WHERE target_binary = 1) AS total_class_1_bad,
        COUNT(*) FILTER (WHERE target_binary = 0) AS total_class_0_good,
        
    FROM services_labeled
    """

print("--- TARGET LABEL STATISTICS ---")
display(con.execute(check_target_query).df())

--- TARGET LABEL STATISTICS ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,exact_global_median,total_class_1_bad,total_class_0_good
0,0.2629,24896,24897


In [71]:
con.execute("""DESCRIBE services_labeled""").df()

,column_name,column_type,null,key,default,extra
0,source,VARCHAR,YES,None,None,None
1,target,VARCHAR,YES,None,None,None
2,YearMonth,VARCHAR,YES,None,None,None
3,year,BIGINT,YES,None,None,None
4,month,BIGINT,YES,None,None,None
...,...,...,...,...,...,...
62,global_median,DOUBLE,YES,None,None,None
63,target_binary,INTEGER,YES,None,None,None
64,target_absolute,INTEGER,YES,None,None,None
65,target_threshold_month_relative,DOUBLE,YES,None,None,None


## Unnest train types
All train types that are under "NS" according to RdT

Thalys == Eurostar merger https://www.eurostar.com/rw-en/about-eurostar/thalys-becomes-eurostar

In [72]:
con.execute("""
SELECT DISTINCT UNNEST(map_keys(train_type_counts)) AS unique_train_type
FROM services_labeled
ORDER BY unique_train_type;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,unique_train_type
0,Eurostar
1,Extra trein
2,ICE International
3,Int. Trein
4,Intercity
5,Intercity direct
6,Speciale Trein
7,Sprinter
8,Thalys


In [ ]:
# if snellius:
con.execute("""
    CREATE OR REPLACE TABLE services_train_ratios AS
    WITH raw_ratios AS (
        SELECT 
            *,
            (COALESCE(train_type_counts['Intercity'], 0)::FLOAT / NULLIF(total_services, 0)) AS raw_intercity,
            (COALESCE(train_type_counts['Intercity direct'], 0)::FLOAT / NULLIF(total_services, 0)) AS raw_intercity_direct,
            ((COALESCE(train_type_counts['Eurostar'], 0) + 
            COALESCE(train_type_counts['Thalys'], 0) + 
            COALESCE(train_type_counts['ICE International'], 0) + 
            COALESCE(train_type_counts['Int. Trein'], 0))::FLOAT / NULLIF(total_services, 0)) AS raw_international,
            ((COALESCE(train_type_counts['Extra trein'], 0) + 
            COALESCE(train_type_counts['Speciale Trein'], 0))::FLOAT / NULLIF(total_services, 0)) AS raw_exceptions
        FROM services_labeled
    )
    SELECT 
        *,
        
        -- 1. Round four of the ratios normally
        ROUND(raw_intercity, 3) AS ratio_intercity,
        ROUND(raw_intercity_direct, 3) AS ratio_intercity_direct,
        ROUND(raw_international, 3) AS ratio_international,
        ROUND(raw_exceptions, 3) AS ratio_operational_exceptions,

        -- 2. Force the Sprinter ratio to absorb the difference, but never drop below 0
        CASE 
            WHEN total_services > 0 THEN 
                GREATEST(0.00, ROUND(1.00 - (
                    ROUND(raw_intercity, 3) + 
                    ROUND(raw_intercity_direct, 3) + 
                    ROUND(raw_international, 3) + 
                    ROUND(raw_exceptions, 3)
                ), 3))
            ELSE NULL 
        END AS ratio_sprinter

    FROM raw_ratios;
""")

if rstats:
    # Quick verification of data types
    print("--- Check Train Type Flattening & Types ---")
    display(con.execute("""
        SELECT 
            total_services, 
            count_intercity, 
            count_sprinter, 
            count_other_trains,
            typeof(count_intercity) as type_check
        FROM services_train_ratios 
        LIMIT 5
    """).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [74]:
con.execute("""SUMMARIZE services_train_ratios""").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,source,VARCHAR,AC,ZZS,291,NaN,NaN,NaN,NaN,NaN,49793,0.0
1,target,VARCHAR,AC,ZZS,280,NaN,NaN,NaN,NaN,NaN,49793,0.0
2,YearMonth,VARCHAR,2019-01,2024-12,84,NaN,NaN,NaN,NaN,NaN,49793,0.0
3,year,BIGINT,2019,2024,6,2021.5182857028096,1.712739114545396,2020,2022,2023,49793,0.0
4,month,BIGINT,1,12,13,6.509268371056173,3.4433126616110923,4,7,10,49793,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
71,ratio_intercity,FLOAT,0.0,1.0,876,0.3734293578067832,0.4609421175773911,0.0,0.01572355124657407,1.0,49793,0.0
72,ratio_intercity_direct,FLOAT,0.0,1.0,333,0.006437210057719168,0.07161505589692992,0.0,0.0,0.0,49793,0.0
73,ratio_international,FLOAT,0.0,1.0,108,0.0030585825331839425,0.05166960340062759,0.0,0.0,0.0,49793,0.0
74,ratio_operational_exceptions,FLOAT,0.0,1.0,189,0.0031904886326166665,0.045026035103419786,0.0,0.0,0.0,49793,0.0


In [ ]:
if rstats:
    display(con.execute("""DESCRIBE services_train_ratios""").df())

## Temporal features

### Month 
- Season ordinal encoding, 
- Month cyclical encoding
https://www.knmi.nl/kennis-en-datacentrum/uitleg/seizoenen

In [ ]:
con.execute("""
    CREATE OR REPLACE VIEW services_temporal_features AS
    SELECT 
        *,
        
        -- Ordinal Season Encoding
        CASE 
            WHEN month IN (12, 1, 2) THEN 1  -- Winter
            WHEN month IN (3, 4, 5) THEN 2   -- Spring
            WHEN month IN (6, 7, 8) THEN 3   -- Summer
            WHEN month IN (9, 10, 11) THEN 4 -- Autumn
        END AS season_ordinal,

        -- Cyclical Month Encoding
        SIN(month * 2 * PI() / 12) AS month_sin,
        COS(month * 2 * PI() / 12) AS month_cos
        
    FROM services_train_ratios
""")

print("✅ View 'services_temporal_features' created.")
print("   - Added ordinal seasons and cyclical sin/cos month features.")

✅ View 'services_temporal_features' created.
   - Added ordinal seasons and cyclical sin/cos month features.


# SAVING FINAL DATASET (with operational features)

DOWNLOAD:  downloads/data/filtered/final_aggregated/final_dataset_with_operational.parquet

In [77]:
display(con.execute("SELECT count(target_binary) FROM services_temporal_features").df())

,count(target_binary)
0,49793


In [78]:
final_with_operational = filtered_output_root_folder / "final_aggregated/final_dataset_with_operational.parquet"
con.execute(f"COPY services_temporal_features TO '{final_with_operational}' (FORMAT PARQUET)")
print("Done! Saved as ", final_with_operational)

Done! Saved as  /home/jbao/NS_Thesis/downloads/data/filtered/final_aggregated/final_dataset_with_operational.parquet


## [Notes] Disruption features
A disruption feature flag based on the affected stations from the disruptions dataset was considered. However, the disruptions dataset has been excluded to prevent data leakage due to the following two reasons: 1. we do not know whether the recorded information (such as affected stations etc) is also known at the recorded start time. 2. we do not know when the disruptions information was made available/announced is aligned with the start time of the disruption. We currently assume this is not the case and therefore decided leave disruptions out.